# ASR + Transcript Q&A — NVIDIA Parakeet v3

> ⚠️ **Runtime → Change runtime type → T4 GPU** before running.

**Pipeline:** Parakeet TDT 0.6B v3 → timestamps → FAISS vector DB → LLaMA 3.1 8B Q&A

**Audio requirement:** 16 kHz mono `.wav` or `.flac`

## Step 1 — Install Dependencies
Run once. The restart cell below will automatically reboot the kernel to apply the numpy pin.

In [ ]:
!pip install -q 'nemo_toolkit[asr]'
!pip install -q sentence-transformers faiss-cpu langchain langchain-nvidia-ai-endpoints
!pip install 'numpy==1.26.4' 'scipy==1.13.1' --force-reinstall -q  # must be last to win
!pip install -U lhotse -q  # fixes TypeError with PyTorch 2.x
print('✓ Done')

In [ ]:
import numpy as np
if np.__version__ != '1.26.4':
    print(f'numpy {np.__version__} → restarting to apply pin...')
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
else:
    print(f'✓ numpy {np.__version__} — ready, continue')

## Step 2 — Imports & API Key

In [ ]:
import os, types
from typing import List, Dict
from sentence_transformers import SentenceTransformer
import faiss
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage, SystemMessage
import nemo.collections.asr as nemo_asr

os.environ['NVIDIA_API_KEY'] = 'YOUR_NVIDIA_API_KEY_HERE'
print('✓ Done')

## Step 3 — Load ASR Model

In [ ]:
from nemo.collections.asr.parts.submodules.transducer_decoding.tdt_label_looping import GreedyBatchedTDTLabelLoopingComputer

asr_model = nemo_asr.models.ASRModel.from_pretrained(model_name='nvidia/parakeet-tdt-0.6b-v3')

# Fix: CUDA 13 changed cudaStreamGetCaptureInfo to return 5 values; NeMo expects 6.
# Monkey-patch so NO_GRAPHS mode is forced after every decoder rebuild.
_NO_GRAPHS = GreedyBatchedTDTLabelLoopingComputer.CudaGraphsMode.NO_GRAPHS
_orig = asr_model.change_decoding_strategy.__func__

def _patched(self, decoding_cfg, verbose=True):
    _orig(self, decoding_cfg, verbose)
    if hasattr(self.decoding, 'decoding') and hasattr(self.decoding.decoding, 'decoding_computer'):
        dc = self.decoding.decoding.decoding_computer
        if hasattr(dc, 'cuda_graphs_mode'):
            dc.cuda_graphs_mode = _NO_GRAPHS

asr_model.change_decoding_strategy = types.MethodType(_patched, asr_model)
asr_model.change_decoding_strategy(asr_model.cfg.decoding, verbose=False)
print('✓ ASR model ready')

## Step 4 — Load Audio & Transcribe

Change `AUDIO_FILE` to point to your own file. Default downloads a sample clip.

In [ ]:
!wget -q -O input.wav https://www.uclass.psychol.ucl.ac.uk/Release2/Conversation/AudioOnly/wav/M_0048_11y6m_2.wav

AUDIO_FILE = 'input.wav'  # ← change to your file if needed

# Optional: uncomment for audio > 24 minutes
# asr_model.change_attention_model(self_attention_model='rel_pos_local_attn', att_context_size=[256, 256])

output = asr_model.transcribe([AUDIO_FILE], timestamps=True)
segment_timestamps = output[0].timestamp['segment']
word_timestamps    = output[0].timestamp['word']

print(f'✓ {len(segment_timestamps)} segments transcribed')
for seg in segment_timestamps[:5]:
    print(f"  [{seg['start']:.2f}s - {seg['end']:.2f}s] {seg['segment']}")

[NeMo I 2026-02-24 00:23:08 nemo_logging:393] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[0][idx].timestep['word'/'segment'/'char']
[NeMo I 2026-02-24 00:23:08 nemo_logging:393] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}


[NeMo W 2026-02-24 00:23:08 nemo_logging:405] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-02-24 00:23:08 nemo_logging:405] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 0it [00:00, ?it/s]


TypeError: Input shape mismatch occured for input_signal in module EncDecRNNTBPEModel : 
Input shape expected = (batch, time) | 
Input shape found : torch.Size([1, 2, 2161319])

## Step 5 — Build Q&A Pipeline

In [24]:
# --- Data model ---
class TranscriptSegment:
    def __init__(self, text: str, start_time: float, end_time: float):
        self.text = text
        self.start_time = start_time
        self.end_time = end_time

    def format_timestamp(self):
        s_m, s_s = divmod(int(self.start_time), 60)
        e_m, e_s = divmod(int(self.end_time), 60)
        return f'{s_m:02d}:{s_s:02d} - {e_m:02d}:{e_s:02d}'

def process_timestamps(segs: List[Dict]) -> List[TranscriptSegment]:
    return [TranscriptSegment(s['segment'], s['start'], s['end']) for s in segs]

# --- Vector DB ---
class TranscriptVectorDB:
    def __init__(self, segments: List[TranscriptSegment]):
        self.segments = segments
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        emb = self.model.encode([s.text for s in segments], show_progress_bar=False)
        self.index = faiss.IndexFlatL2(emb.shape[1])
        self.index.add(emb.astype('float32'))

    def search(self, query: str, k: int = 5):
        k = min(k, len(self.segments))
        q = self.model.encode([query]).astype('float32')
        dists, idxs = self.index.search(q, k)
        return [(self.segments[i], float(d)) for i, d in zip(idxs[0], dists[0])]

# --- Q&A ---
class TranscriptQA:
    def __init__(self, db: TranscriptVectorDB, llm):
        self.db = db
        self.llm = llm

    def ask(self, question: str, top_k: int = 5) -> str:
        hits = self.db.search(question, k=top_k)
        context = '\n\n'.join(
            f'Segment {i+1} [{seg.format_timestamp()}]:\n{seg.text}'
            for i, (seg, _) in enumerate(hits)
        )
        msgs = [
            SystemMessage(content='Answer based only on the transcript segments. Reference timestamps. Be concise.'),
            HumanMessage(content=f'Question: {question}\n\nSegments:\n{context}')
        ]
        answer = self.llm.invoke(msgs).content
        sources = '\n'.join(f'  [{s.format_timestamp()}] {s.text}' for s, _ in hits)
        return f'{answer}\n\n📍 Sources:\n{sources}'

# --- Build ---
transcript_segments = process_timestamps(segment_timestamps)
vector_db = TranscriptVectorDB(transcript_segments)

llm = ChatNVIDIA(
    model='meta/llama-3.1-8b-instruct',
    nvidia_api_key=os.environ.get('NVIDIA_API_KEY'),
    temperature=0.2,
    max_tokens=500
)

qa = TranscriptQA(vector_db, llm)
print(f'✓ Q&A ready — {len(transcript_segments)} segments indexed')

✓ Q&A ready — 86 segments indexed


## Step 6 — Ask a Question

Edit `question` and re-run this cell.

In [27]:
question = 'Summarize the conversation'  # ← edit your question here

print(f'❓ {question}\n')
print(qa.ask(question))

❓ Summarize the conversation

There is no conversation. The provided segments appear to be disjointed and unrelated statements.

📍 Sources:
  [02:11 - 02:12] Describe some of the stuff.
  [03:52 - 04:13] Um what about um about describe your bedroom?
  [01:32 - 01:32] I'm very sorry.
  [02:27 - 02:43] Oh what about er describe your journey here?
  [02:20 - 02:25] Okay, what about uh have you learned anything interesting in history?
